# 🦊 Bomi (보미) — 아이 영어 회화 AI 에이전트

---

## Step 1: 에이전트 설계

### 이름
**Bomi (보미)** — 호기심 많은 아기 여우 캐릭터

### 목적
한국 아이(5~10세)가 영어로 "말하기"를 연습할 수 있는 AI 대화 파트너.  
선생님이 아닌 **친구**로서 자연스러운 영어 회화를 유도한다.

### 핵심 기능
1. **캐릭터 대화 (Character Conversation)** — 보미 캐릭터의 성격·말투로 아이와 자유 영어 대화
2. **자연스러운 교정 (Gentle Correction)** — 아이의 문법 오류를 자연스럽게 반복·교정
3. **오늘의 표현 (Daily Expression)** — 세션마다 표현 1개를 자연스럽게 유도하고 사용 여부 추적
4. **대화 분석 (Conversation Analysis)** — 세션 종료 후 사용 단어, 교정 횟수, 표현 사용 여부 요약
5. **지식 검색 (Knowledge Search)** 🆕 — 아이가 궁금한 주제(동물, 우주, 음식 등)를 웹 검색으로 찾아 재미있는 사실 제공

### 그래프 구조

```
                    ┌─────────────┐
                    │   START     │
                    └──────┬──────┘
                           │
                           ▼
                ┌──────────────────┐
                │  session_setup   │  세션 초기화: 연령대, 오늘의 표현 선택,
                │                  │  보미 인사 메시지 생성
                └────────┬─────────┘
                         │
                         ▼
                ┌──────────────────┐
            ┌──▶│ input_analyzer   │◀──────────────────────┐
            │   │  (interrupt)     │  아이 입력 분석        │
            │   └────────┬─────────┘                       │
            │            │                                 │
            │    route_by_analysis                         │
            │     (conditional)                            │
            │      ╱          ╲                            │
            │     ╱            ╲                           │
            │    ▼              ▼                          │
            │ ┌────────┐  ┌──────────────────┐            │
            │ │ normal │  │ knowledge_search │ 🔍 Tool    │
            │ └───┬────┘  │ (웹 검색 수행)    │            │
            │     │       └────────┬─────────┘            │
            │     │                │                       │
            │     ▼                ▼                       │
            │   ┌──────────────────┐                      │
            │   │  bomi_respond    │  보미 캐릭터로 응답    │
            │   │                  │  (검색 결과 반영)      │
            │   └────────┬─────────┘                      │
            │            │                                 │
            │    should_continue                           │
            │     (conditional)                            │
            │      ╱          ╲                            │
            │  (계속)        (종료)                         │
            │     │            │                           │
            └─────┘            ▼                           │
                      ┌──────────────────┐                │
                      │ session_summary  │  대화 분석 리포트│
                      └────────┬─────────┘                │
                               │                           │
                               ▼                           │
                         ┌──────────┐                      │
                         │   END    │                      │
                         └──────────┘
```

**노드 설명 (5개):**
- `session_setup`: 연령대에 따른 프롬프트 설정 + 오늘의 표현 선택 + 보미 첫 인사 생성
- `input_analyzer`: 아이 입력을 분석하여 주제 검색 필요 여부 판단 (interrupt point)
- `knowledge_enrichment`: 🔍 **Tool 사용** — Tavily 웹 검색으로 아이가 궁금한 주제의 재미있는 사실 검색
- `bomi_respond`: 보미 캐릭터로 LLM 대화 수행 (검색 결과가 있으면 반영)
- `session_summary`: 대화 내용 분석 후 부모 리포트용 요약 생성

**조건부 엣지 (2개):**
- `route_by_analysis`: 입력 분석 결과에 따라 검색 필요 → `knowledge_enrichment`, 일반 대화 → `bomi_respond`
- `should_continue`: 종료 조건 판단 (사용자 종료 / 세션 시간 초과)

---

## Step 2: 기초 구축 — LangGraph 구현

In [ ]:
import os
import random
from typing import Annotated, Literal

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

### State 정의

대화 메시지 외에 세션 메타데이터(연령대, 오늘의 표현, 턴 수)를 함께 관리한다.

In [ ]:
class BomiState(TypedDict):
    """보미 에이전트 상태"""
    messages: Annotated[list, add_messages]  # 대화 히스토리
    age_group: str           # "5-7" or "8-10"
    daily_expression: str    # 오늘의 표현
    turn_count: int          # 대화 턴 수
    session_active: bool     # 세션 활성 여부
    search_query: str        # 검색이 필요한 주제 (없으면 빈 문자열)
    search_result: str       # 검색 결과 (bomi_respond에서 참고)

### 보미 시스템 프롬프트 & 설정

In [ ]:
DAILY_EXPRESSIONS = [
    "delicious",
    "excited",
    "I want to...",
    "my favorite",
    "Let's go to...",
]


def build_system_prompt(age_group: str, daily_expression: str) -> str:
    """연령대와 오늘의 표현에 맞는 시스템 프롬프트 생성"""

    if age_group == "5-7":
        vocab_note = "Use only simple words (about 300-word vocabulary: animals, food, colors, family, toys). Keep responses to 1-2 very short sentences."
        tone_note = "Be VERY enthusiastic. Use lots of 'Wow!', 'Yay!', 'So cool!' reactions."
        topics = "animals, food, games, family, cartoons"
    else:
        vocab_note = "Use age-appropriate vocabulary (about 800 words: school, hobbies, travel, emotions). Keep responses to 1-2 sentences."
        tone_note = "Be a curious, fun friend. React with genuine interest."
        topics = "school, hobbies, sports, movies, dreams, travel"

    return f"""You are Bomi (보미), a curious little fox who loves talking with kids in English.

[IDENTITY]
- Name: Bomi, a baby fox
- Personality: Extremely curious, asks about everything, sometimes makes silly mistakes
- Quirk: You occasionally "forget" a word and say "um... what's that called..." so the child can help you
- Catchphrases: "Ohhh!", "Wait wait wait!", "That's SO cool!"
- You always start with: "Hey! It's Bomi! I missed you!"
- You always end with: "See you tomorrow! Bye bye!"

[LANGUAGE]
- Target age group: {age_group} years old
- {vocab_note}
- {tone_note}
- Preferred topics: {topics}

[CONVERSATION RULES]
- You are a FRIEND, not a teacher. Never say "let me teach you" or "the correct way is".
- Always end your response with a question to keep the conversation going.
- Maximum 2 sentences per response.
- Accept single-word answers enthusiastically. "Dog" → "You like dogs? Me too! What's your dog's name?"
- If the child speaks Korean, gently respond in English: "Ohhh! In English, we say [translation]! That's a fun word!"

[CORRECTION]
- Never explicitly correct grammar. Instead, naturally repeat the correct form.
- Example: Child says "I goed to school" → You say "Oh, you went to school! What did you do there?"
- If the child's pronunciation seems off (based on text), model the correct word naturally.

[DAILY EXPRESSION]
- Today's expression: "{daily_expression}"
- Naturally weave this expression into the conversation early on.
- If the child uses it, react with extra excitement: "Ohhh! You said '{daily_expression}'! That's AMAZING!"

[SAFETY]
- Never ask for personal information (real name, address, school name, phone number).
- If the child shares personal info, redirect: "That's nice! Let's talk about something fun instead!"
- Never pretend to be human.
- Avoid scary, violent, or inappropriate topics. Keep everything positive and fun.
- If the child is silent for too long, gently prompt: "Hey, are you still there? I have a fun question for you!"
"""

### LLM 설정 & Tool 정의

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

# --- Tool: 아이가 궁금한 주제를 웹에서 검색 ---

@tool
def search_topic_for_kids(query: str) -> str:
    """Search for fun, kid-friendly facts about a topic.
    Use this when the child asks about animals, space, food, science, or any curious topic.
    Returns simple, exciting facts suitable for children aged 5-10.
    """
    search = TavilySearchResults(max_results=2)
    results = search.invoke(f"{query} fun facts for kids simple")

    if not results:
        return f"Fun fact: {query} is a really interesting topic!"

    # 검색 결과를 간단하게 정리
    facts = []
    for r in results[:2]:
        content = r.get("content", "")
        if content:
            # 첫 200자만 추출 (아이용으로 간결하게)
            facts.append(content[:200])

    return " | ".join(facts) if facts else f"Fun fact: {query} is a really interesting topic!"


tools = [search_topic_for_kids]

print("✅ LLM and tools configured!")

### 노드 구현 (5개)

1. `session_setup` — 세션 초기화
2. `input_analyzer` — 아이 입력 분석 (검색 필요 여부 판단)
3. `knowledge_enrichment` — 🔍 Tool로 웹 검색 수행
4. `bomi_respond` — 보미 캐릭터로 응답 생성
5. `session_summary` — 부모 리포트 생성

In [ ]:
# ── Node 1: session_setup ──

def session_setup(state: BomiState) -> BomiState:
    """세션 초기화: 오늘의 표현 선택 + 보미 첫 인사 생성"""
    age_group = state.get("age_group", "5-7")
    daily_expression = random.choice(DAILY_EXPRESSIONS)
    system_prompt = build_system_prompt(age_group, daily_expression)

    setup_messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content="[SESSION START] The child just opened the app. Greet them as Bomi and introduce today's expression naturally."),
    ]
    response = llm.invoke(setup_messages)

    return {
        "messages": [SystemMessage(content=system_prompt), AIMessage(content=response.content)],
        "daily_expression": daily_expression,
        "turn_count": 0,
        "session_active": True,
        "search_query": "",
        "search_result": "",
    }


# ── Node 2: input_analyzer ──

def input_analyzer(state: BomiState) -> BomiState:
    """아이의 입력을 분석하여 주제 검색이 필요한지 판단"""
    last_message = state["messages"][-1]

    # HumanMessage가 아니면 (첫 진입 등) 검색 불필요
    if not isinstance(last_message, HumanMessage):
        return {"search_query": ""}

    text = last_message.content

    # LLM을 사용하여 아이가 궁금해하는 주제가 있는지 분석
    analysis_messages = [
        SystemMessage(content="""You analyze a child's message in an English conversation.
If the child is asking about or mentioning a specific topic where a fun fact would make
the conversation more engaging (e.g., animals, dinosaurs, space, food, countries, sports),
extract the main topic as a single word or short phrase.

Rules:
- If the child asks "what is...", "tell me about...", "do you know...", "I like [topic]" → extract the topic
- If it's just casual chat, greetings, or short responses like "yes", "no", "okay" → return "none"
- Return ONLY the topic word/phrase or "none". No explanation."""),
        HumanMessage(content=f"Child's message: {text}"),
    ]
    response = llm.invoke(analysis_messages)
    topic = response.content.strip().strip('"').lower()

    if topic and topic != "none":
        return {"search_query": topic}
    return {"search_query": ""}


# ── Node 3: knowledge_enrichment ──

def knowledge_enrichment(state: BomiState) -> BomiState:
    """🔍 Tool 사용: 아이가 궁금한 주제를 웹 검색으로 조회"""
    query = state.get("search_query", "")
    if not query:
        return {"search_result": ""}

    try:
        result = search_topic_for_kids.invoke({"query": query})
        return {"search_result": result}
    except Exception as e:
        print(f"⚠️ Search failed: {e}")
        return {"search_result": ""}


# ── Node 4: bomi_respond ──

def bomi_respond(state: BomiState) -> BomiState:
    """보미 캐릭터로 대화 응답 생성 (검색 결과가 있으면 반영)"""
    messages = list(state["messages"])
    search_result = state.get("search_result", "")

    # 검색 결과가 있으면 시스템 힌트로 추가
    if search_result:
        search_hint = SystemMessage(
            content=f"""[KNOWLEDGE HINT] You just looked up something interesting!
Here's what you found: {search_result}

Share ONE simple, exciting fun fact with the child in your Bomi character.
Keep it very short (1-2 sentences) and age-appropriate.
React with excitement like "Ohhh! Did you know that...?!"
Then ask a follow-up question to keep the conversation going."""
        )
        messages.append(search_hint)

    response = llm.invoke(messages)

    return {
        "messages": [response],
        "turn_count": state["turn_count"] + 1,
        "search_query": "",    # 사용 후 초기화
        "search_result": "",   # 사용 후 초기화
    }


# ── Node 5: session_summary ──

def session_summary(state: BomiState) -> BomiState:
    """세션 종료 후 대화 분석 요약 (부모 리포트용)"""
    daily_expr = state.get("daily_expression", "")

    summary_prompt = f"""Analyze the conversation above between Bomi (AI fox) and a child.
Provide a brief summary in Korean for the parent report:

1. 총 대화 턴 수: {state['turn_count']}
2. 아이가 사용한 영어 단어/표현 목록
3. 교정이 이루어진 부분 (있다면)
4. 오늘의 표현 "{daily_expr}" 사용 여부
5. 전체 평가 (한 줄)

Format as a clean parent-friendly report in Korean."""

    messages = state["messages"] + [HumanMessage(content=summary_prompt)]
    response = llm.invoke(messages)

    return {
        "messages": [AIMessage(content=f"[부모 리포트]\n{response.content}")],
        "session_active": False,
    }

### 조건부 엣지 (2개)

1. `route_by_analysis` — 입력 분석 결과에 따라 검색 or 일반 대화로 분기
2. `should_continue` — 대화 계속 여부 판단 (종료 키워드 / 최대 턴)

In [ ]:
MAX_TURNS = 10  # 최대 턴 수


# ── Conditional Edge 1: route_by_analysis ──

def route_by_analysis(state: BomiState) -> Literal["knowledge_enrichment", "bomi_respond"]:
    """입력 분석 결과에 따라 검색이 필요하면 knowledge_enrichment로, 아니면 bomi_respond로 분기"""
    if state.get("search_query", ""):
        return "knowledge_enrichment"
    return "bomi_respond"


# ── Conditional Edge 2: should_continue ──

def should_continue(state: BomiState) -> Literal["input_analyzer", "session_summary"]:
    """대화 계속 여부 판단"""
    # 마지막 메시지가 종료 신호인지 확인
    last_message = state["messages"][-1]
    if isinstance(last_message, HumanMessage):
        text = last_message.content.lower()
        if any(word in text for word in ["bye", "quit", "exit", "끝", "그만"]):
            return "session_summary"

    # 최대 턴 수 초과
    if state["turn_count"] >= MAX_TURNS:
        return "session_summary"

    return "input_analyzer"

### 그래프 조립

In [ ]:
# 그래프 빌드
graph_builder = StateGraph(BomiState)

# 노드 추가 (5개)
graph_builder.add_node("session_setup", session_setup)
graph_builder.add_node("input_analyzer", input_analyzer)
graph_builder.add_node("knowledge_enrichment", knowledge_enrichment)
graph_builder.add_node("bomi_respond", bomi_respond)
graph_builder.add_node("session_summary", session_summary)

# 엣지 연결
graph_builder.add_edge(START, "session_setup")
graph_builder.add_edge("session_setup", "input_analyzer")

# 조건부 엣지 1: 입력 분석 → 검색 or 바로 응답
graph_builder.add_conditional_edges("input_analyzer", route_by_analysis)

# knowledge_enrichment → bomi_respond (검색 후 응답 생성)
graph_builder.add_edge("knowledge_enrichment", "bomi_respond")

# 조건부 엣지 2: 응답 후 계속 or 종료
graph_builder.add_conditional_edges("bomi_respond", should_continue)

# 세션 요약 → END
graph_builder.add_edge("session_summary", END)

# 컴파일 (input_analyzer 진입 전에 interrupt → 사용자 입력 대기)
graph = graph_builder.compile(interrupt_before=["input_analyzer"])

print("✅ Bomi agent graph compiled successfully!")
print("   Nodes: session_setup → input_analyzer → (knowledge_enrichment →) bomi_respond → session_summary")
print("   Conditional Edges: route_by_analysis, should_continue")
print("   Tool: search_topic_for_kids (Tavily web search)")

### 그래프 시각화

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # fallback: mermaid 텍스트 출력
    print(graph.get_graph().draw_mermaid())

### 대화 테스트 실행

Human-in-the-loop 방식으로 실제 대화를 시뮬레이션한다.  
`interrupt_before=["conversation"]`으로 매 턴마다 사용자 입력을 받는다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 체크포인터로 리컴파일 (human-in-the-loop에 필요)
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory, interrupt_before=["input_analyzer"])

config = {"configurable": {"thread_id": "demo-session-1"}}

# 1) 세션 시작 — session_setup 실행 후 input_analyzer 직전에서 멈춤
initial_state = {"age_group": "5-7"}
state = graph.invoke(initial_state, config)

# 보미 인사 출력
for msg in state["messages"]:
    if isinstance(msg, AIMessage):
        print(f"🦊 Bomi: {msg.content}")

print(f"\n📝 오늘의 표현: {state['daily_expression']}")
print("─" * 50)

In [ ]:
# 2) 대화 루프 — 사용자 입력을 받아 대화 진행
#    'bye'를 입력하면 세션 종료 → 부모 리포트 생성
#    아이가 특정 주제(동물, 우주 등)를 언급하면 자동으로 웹 검색 후 재미있는 사실 제공!

while True:
    user_input = input("👧 You: ")
    if not user_input.strip():
        continue

    # 사용자 메시지를 state에 추가하고 그래프 재개
    state = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config,
    )

    # 최신 AI 메시지 출력
    last_ai = [m for m in state["messages"] if isinstance(m, AIMessage)][-1]
    print(f"🦊 Bomi: {last_ai.content}")

    # 검색이 실행되었으면 표시
    if state.get("search_query"):
        print(f"   🔍 (searched: {state['search_query']})")

    # 세션 종료 확인
    if not state.get("session_active", True):
        print("\n" + "═" * 50)
        print("세션이 종료되었습니다.")
        break

    print("─" * 50)